## Pobranie danych o 50 gwiazdach

In [ ]:
from astroquery.gaia import Gaia
from astropy import units as u
from astropy.table import Table

# Make sure we are using the right DR3 table
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

# Zapytanie ADQL:
# - take the 50 objects with the largest parallax (parallax in mas)
# - pomijamy parallax <= 0, bo to bez sensu fizycznie
query = """
SELECT
    TOP 50
    g.source_id,
    g.ra, g.dec,
    g.parallax,                     -- w mili-arcsekundach
    g.pmra, g.pmdec,                -- proper motions [mas/yr]
    g.radial_velocity,              -- km/s (often NULL)
    g.phot_g_mean_mag,
    ap.mass_flame,                  -- masa gwiazdy [Msun] (modelowana)
    ap.radius_flame                 -- radius [Rsun] (modelled)
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.astrophysical_parameters AS ap
  ON g.source_id = ap.source_id
WHERE g.parallax > 0
ORDER BY g.parallax DESC
"""

job = Gaia.launch_job_async(query)
tbl = job.get_results()

# Add a distance column in parsecs: d[pc] = 1000 / parallax[mas]
distance_pc = (1000.0 / tbl["parallax"]) * u.pc
tbl["distance_pc"] = distance_pc

# You can inspect the first rows
tbl[:5].pprint(max_width=120)

# Write to CSV, e.g. for further processing / REBOUND
tbl.write("nearest_50_gaia.csv", format="csv", overwrite=True)
print("Zapisano do nearest_50_gaia.csv")

## Building the Solar System, assigning masses and radii

In [ ]:
import os
import urllib.request

import numpy as np
import rebound
import spiceypy as sp

from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from astropy.constants import M_sun, R_sun, R_earth, R_jup, G

# ------------------------------------------------------------
# 1. SPICE configuration (Solar System ephemerides)
# ------------------------------------------------------------
KERNEL_DIR = "kernels"
os.makedirs(KERNEL_DIR, exist_ok=True)

DE440_URL = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440.bsp"
LSK_URL = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls"
GM_PCK_URL = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/gm_de431.tpc"
RADII_PCK_URL = "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00010.tpc"


def download_if_needed(url: str, path: str) -> None:
    if not os.path.exists(path):
        print(f"Pobieram kernel z {url} -> {path}")
        urllib.request.urlretrieve(url, path)


de440_path = os.path.join(KERNEL_DIR, "de440.bsp")
lsk_path = os.path.join(KERNEL_DIR, "naif0012.tls")
gm_pck_path = os.path.join(KERNEL_DIR, "gm_de431.tpc")
radii_pck_path = os.path.join(KERNEL_DIR, "pck00010.tpc")

download_if_needed(DE440_URL, de440_path)
download_if_needed(LSK_URL, lsk_path)
download_if_needed(GM_PCK_URL, gm_pck_path)
download_if_needed(RADII_PCK_URL, radii_pck_path)

sp.furnsh(lsk_path)
sp.furnsh(de440_path)
sp.furnsh(gm_pck_path)
sp.furnsh(radii_pck_path)

# ------------------------------------------------------------
# 2. Pick an epoch (datetime) and convert it to SPICE time
# ------------------------------------------------------------
# Change the epoch here if you want something other than the start of 2025
obs_time = Time("2026-01-01T00:00:00", scale="utc")
et = sp.utc2et(obs_time.utc.isot)

# ------------------------------------------------------------
# 3. Helpers: units and body state from SPICE
# ------------------------------------------------------------
KM_TO_AU = (1 * u.km).to(u.au).value
SEC_PER_YEAR = (1 * u.yr).to(u.s).value


def spice_state_wrt_sun(target: str, et: float):
    """Return (r, v) relative to the Sun in REBOUND units (AU, AU/yr)."""
    state, _ = sp.spkezr(target, et, "ECLIPJ2000", "NONE", "SUN")
    r_km = np.array(state[:3])
    v_kms = np.array(state[3:])

    r_au = r_km * KM_TO_AU
    v_au_per_yr = v_kms * KM_TO_AU * SEC_PER_YEAR
    return r_au, v_au_per_yr


# ------------------------------------------------------------
# 4. Load the 50 nearest stars from the Gaia file
# ------------------------------------------------------------
# The file should have been generated by the first cell
stars_tbl = Table.read("nearest_50_gaia.csv", format="csv")

print(f"Liczba gwiazd (bez filtrowania): {len(stars_tbl)}")

# Build coordinates: RA/DEC + distance (distance_pc)
coords = SkyCoord(
    ra=stars_tbl["ra"] * u.deg,
    dec=stars_tbl["dec"] * u.deg,
    distance=stars_tbl["distance_pc"] * u.pc,
)

# Cartesian coordinates relative to the Sun
cart = coords.cartesian
x_au = cart.x.to(u.au).value
y_au = cart.y.to(u.au).value
z_au = cart.z.to(u.au).value

# ------------------------------------------------------------
# Stellar velocities
# ------------------------------------------------------------
# 1) Initially 0 for everything (fallback for stars without proper motion)
vx_auyr = np.zeros(len(stars_tbl))
vy_auyr = np.zeros(len(stars_tbl))
vz_auyr = np.zeros(len(stars_tbl))

# 2) For stars with proper motions (pmra, pmdec), compute velocities
pmra = np.array(stars_tbl["pmra"], dtype=float)
pmdec = np.array(stars_tbl["pmdec"], dtype=float)
rv = np.array(stars_tbl["radial_velocity"], dtype=float)

mask_pm = (~np.isnan(pmra)) & (~np.isnan(pmdec))

if mask_pm.any():
    # If radial_velocity is NaN we take rv = 0 km/s (no radial component),
    # but the tangential velocity from proper motions is still computed.
    rv_eff = np.where(np.isnan(rv), 0.0, rv)

    coords_vel = SkyCoord(
        ra=stars_tbl["ra"][mask_pm] * u.deg,
        dec=stars_tbl["dec"][mask_pm] * u.deg,
        distance=stars_tbl["distance_pc"][mask_pm] * u.pc,
        pm_ra_cosdec=pmra[mask_pm] * u.mas / u.yr,
        pm_dec=pmdec[mask_pm] * u.mas / u.yr,
        radial_velocity=rv_eff[mask_pm] * u.km / u.s,
    )

    cart_vel = coords_vel.cartesian
    if "s" in cart_vel.differentials:
        diff = cart_vel.differentials["s"]  # CartesianDifferential
        vx_auyr[mask_pm] = diff.d_x.to(u.au / u.yr).value
        vy_auyr[mask_pm] = diff.d_y.to(u.au / u.yr).value
        vz_auyr[mask_pm] = diff.d_z.to(u.au / u.yr).value
    else:
        print("Note: no velocity differentials in the Cartesian representation.")

    print(f"Stars with non-zero velocity (have pm): {mask_pm.sum()}")
else:
    print("No stars with proper motion data; all have v=0.")

# ------------------------------------------------------------
# 5. Initialise the REBOUND simulation (Solar System from JPL Horizons)
# ------------------------------------------------------------
sim = rebound.Simulation()
# Units used: AU, year, solar mass (Msun)
sim.units = ("AU", "yr", "Msun")

# Data/epoka jako string dla interfejsu Horizons w REBOUND
date_str = obs_time.utc.datetime.strftime("%Y-%m-%d %H:%M")

# Add the Sun and planets directly from JPL Horizons.
# REBOUND fetches ephemerides, masses and radii itself (where available).
sim.add("Sun")

planet_names = [
    "Mercury",
    "Venus",
    "Earth",
    "Mars",
    "Jupiter",
    "Saturn",
    "Uranus",
    "Neptune",
]

for body in planet_names:
    sim.add(body, date=date_str)

# Set Sun and planet radii from astropy (and simple coefficients)
# REBOUND/Horizons does not always assign a sensible radius, so we do it explicitly.
Rsun_AU = R_sun.to(u.au).value
Rearth_AU = R_earth.to(u.au).value
Rjup_AU = R_jup.to(u.au).value

radii_planets_AU = {
    "Sun": Rsun_AU,
    # ~IAU/NASA coefficients relative to Earth / Jupiter radius
    "Mercury": 0.383 * Rearth_AU,
    "Venus":   0.950 * Rearth_AU,
    "Earth":   1.000 * Rearth_AU,
    "Mars":    0.532 * Rearth_AU,
    "Jupiter": 1.000 * Rjup_AU,
    "Saturn":  0.843 * Rjup_AU,
    "Uranus":  0.358 * Rjup_AU,
    "Neptune": 0.346 * Rjup_AU,
}

# Indices: 0 = Sun, 1.. = planets in planet_names order
sim.particles[0].r = radii_planets_AU["Sun"]
for i, body in enumerate(planet_names, start=1):
    sim.particles[i].r = radii_planets_AU[body]

# Keep the planet name list for later use (e.g. in the summary)
solar_system_bodies = planet_names

# ------------------------------------------------------------
# 6. Add the 50 nearest stars as extra point masses
# ------------------------------------------------------------
# Gaia DR3 stellar masses (mass_flame) are given in Msun where available.
# Where masses are missing we estimate them from the absolute G-band magnitude.

mass_flame = np.array(stars_tbl["mass_flame"], dtype=float)
radius_flame = np.array(stars_tbl["radius_flame"], dtype=float)

star_masses = np.full(len(stars_tbl), np.nan)
star_radii_Rsun = np.full(len(stars_tbl), np.nan)

# Only positive Gaia masses/radii are used; zeros are treated as missing data
mask_mass = np.isfinite(mass_flame) & (mass_flame > 0)
mask_radius = np.isfinite(radius_flame) & (radius_flame > 0)

star_masses[mask_mass] = mass_flame[mask_mass]
star_radii_Rsun[mask_radius] = radius_flame[mask_radius]

# Estimate missing masses and radii from G-band photometry
missing_mass = ~np.isfinite(star_masses)
missing_radius = ~np.isfinite(star_radii_Rsun)

if missing_mass.any() or missing_radius.any():
    Gmag = np.array(stars_tbl["phot_g_mean_mag"], dtype=float)
    d_pc = np.array(stars_tbl["distance_pc"], dtype=float)
    # Absolute G-band magnitude: M_G = m_G - 5 log10(d/10 pc)
    M_G = Gmag - 5.0 * (np.log10(d_pc) - 1.0)

    # We take M_G(Sun) ~ 4.67
    M_G_sun = 4.67
    L_over_Lsun = 10.0 ** (-0.4 * (M_G - M_G_sun))

    # Main-sequence mass-luminosity relation: L ~ M^3.5 => M ~ L^(1/3.5)
    M_over_Msun_est = L_over_Lsun ** (1.0 / 3.5)
    M_over_Msun_est = np.clip(M_over_Msun_est, 0.08, 20.0)

    # Simple radius-mass relation: R ~ M^0.8 (in Rsun)
    R_over_Rsun_est = M_over_Msun_est ** 0.8
    R_over_Rsun_est = np.clip(R_over_Rsun_est, 0.1, 20.0)

    star_masses[missing_mass] = M_over_Msun_est[missing_mass]
    star_radii_Rsun[missing_radius] = R_over_Rsun_est[missing_radius]

# Just in case - if anything is still NaN, set 1 Msun and 1 Rsun
star_masses = np.where(np.isfinite(star_masses), star_masses, 1.0)
star_radii_Rsun = np.where(np.isfinite(star_radii_Rsun), star_radii_Rsun, 1.0)

# Konwersja promieni z Rsun na AU
Rsun_AU = R_sun.to(u.au).value
star_radii_AU = star_radii_Rsun * Rsun_AU

print(f"Liczba gwiazd z mass_flame (Gaia): {mask_mass.sum()}")
print(f"Liczba gwiazd z radius_flame (Gaia): {mask_radius.sum()}")

for i in range(len(stars_tbl)):
    sim.add(
        m=star_masses[i],
        r=star_radii_AU[i],
        x=x_au[i],
        y=y_au[i],
        z=z_au[i],
        vx=vx_auyr[i],
        vy=vy_auyr[i],
        vz=vz_auyr[i],
    )

print(f"Particles in the simulation: {sim.N}")
print("Particle 0: Sun, 1-8: planets, 9-...: nearest Gaia stars.")

# Example short forward integration (e.g. 1 year)
# sim.integrate(1.0)
# print("Integrated the simulation forward by 1 year.")

## Sanity check

In [ ]:
import numpy as np
import pandas as pd
from astropy.table import Table

# 3. Summary: position, mass and size of every body in the simulation
# Assumes cells 0 (Gaia) and 1 (REBOUND+SPICE) have already been run

# Load the star table so we have their identifiers
stars_tbl = Table.read("nearest_50_gaia.csv", format="csv")

rows = []

# REBOUND indices: 0 = Sun, 1-8 = planets, 9+ = stars

# Sun
p0 = sim.particles[0]
rows.append({
    "name": "SUN",
    "kind": "star_sun",
    "mass_Msun": p0.m,
    "radius_AU": p0.r,
    "x_AU": p0.x,
    "y_AU": p0.y,
    "z_AU": p0.z,
})

# Planety
for i, body in enumerate(solar_system_bodies, start=1):
    p = sim.particles[i]
    rows.append({
        "name": body,
        "kind": "planet",
        "mass_Msun": p.m,
        "radius_AU": p.r,
        "x_AU": p.x,
        "y_AU": p.y,
        "z_AU": p.z,
    })

# Gwiazdy z Gaia
offset = 1 + len(solar_system_bodies)
for j in range(len(stars_tbl)):
    p = sim.particles[offset + j]
    sid = int(stars_tbl["source_id"][j])
    rows.append({
        "name": f"GAIA_DR3_{sid}",
        "kind": "star_gaia",
        "mass_Msun": p.m,
        "radius_AU": p.r,
        "x_AU": p.x,
        "y_AU": p.y,
        "z_AU": p.z,
    })

bodies_df = pd.DataFrame(rows)

# Table preview
print(bodies_df.head(12))
print("\nType summary:")
print(bodies_df["kind"].value_counts())

## Moving to the barycentric frame

In [ ]:
import numpy as np

# 4. Move to the barycentric frame (optional, but recommended)
# Assumes cell 1 has been run and the `sim` object already contains
# the Sun, the planets and the 50 Gaia stars.

# Barycentre and total momentum BEFORE the transform (for reference)
M_tot = 0.0
R_cm = np.zeros(3)
P_tot = np.zeros(3)

for p in sim.particles:
    m = p.m
    r_vec = np.array([p.x, p.y, p.z])
    v_vec = np.array([p.vx, p.vy, p.vz])
    M_tot += m
    R_cm += m * r_vec
    P_tot += m * v_vec

R_cm /= M_tot
V_cm = P_tot / M_tot

print("Przed move_to_com():")
print(f"  R_cm = {R_cm} AU")
print(f"  V_cm = {V_cm} AU/yr")

# Shift to the barycentric frame
sim.move_to_com()

# Barycentre and momentum AFTER the transform (should be ~0)
M_tot2 = 0.0
R_cm2 = np.zeros(3)
P_tot2 = np.zeros(3)

for p in sim.particles:
    m = p.m
    r_vec = np.array([p.x, p.y, p.z])
    v_vec = np.array([p.vx, p.vy, p.vz])
    M_tot2 += m
    R_cm2 += m * r_vec
    P_tot2 += m * v_vec

R_cm2 /= M_tot2
V_cm2 = P_tot2 / M_tot2

print("\nPo move_to_com():")
print(f"  R_cm = {R_cm2} AU")
print(f"  V_cm = {V_cm2} AU/yr")

## Beta parameter helpers for radiation pressure

In [ ]:
import numpy as np
from astropy.constants import G as G_SI, M_sun as M_SUN_SI, L_sun as L_SUN_SI, c as C_SI
import astropy.units as u

# 5. Radiation pressure helpers
# -------------------------------------------------
# Assumptions:
# - radiation pressure acts ONLY on asteroids (low-mass test particles),
# - in each step we only account for the nearest star (the Sun or a Gaia star),
# - stellar luminosity L_* is estimated from mass: L/L_sun ~ (M/M_sun)^3.5.


def compute_beta_single_star(M_star_Msun, R_body_m, rho=2000.0, Q_pr=1.0):
    """Return beta = Frad/Fgrav for a small body near a star.

    Accepts both scalars and numpy arrays (R_body_m, rho, Q_pr may be arrays,
    in which case the result is an array too).

    Parametry
    ---------
    M_star_Msun : float — masa gwiazdy [Msun]
    R_body_m    : float | array - body (asteroid) radius [m]
    rho         : float | array - body density [kg/m^3]
    Q_pr        : float | array - radiation pressure efficiency coefficient

    Zwraca
    -------
    beta : float | array - dimensionless ratio of radiation force to gravity.
    """
    L_star_Lsun = M_star_Msun ** 3.5

    L_star_SI = L_star_Lsun * L_SUN_SI.value
    M_star_SI = M_star_Msun * M_SUN_SI.value

    beta = (3.0 * L_star_SI * Q_pr) / (16.0 * np.pi * C_SI.value * G_SI.value * M_star_SI * rho * R_body_m)
    return beta


def radiation_pressure_accel_nearest_star(
    sim,
    i_ast: int,
    star_indices,
    R_body_m: float,
    rho: float = 2000.0,
    Q_pr: float = 1.0,
):
    """Compute the radiation pressure acceleration for a given asteroid.

    - Finds the nearest star from star_indices (e.g. [0] + Gaia star indices).
    - Dla tej gwiazdy liczy beta (Frad/Fgrav) z prostego wzoru.
    - Z beta i grawitacji tej gwiazdy liczy wektor przyspieszenia od promieniowania.

    Zwraca
    -------
    a_rad_vec : np.array(3) w jednostkach REBOUND (AU/yr^2)
    beta      : dimensionless radiation pressure coefficient for this asteroid
    j_near    : index of the nearest star in sim.particles
    """
    p_ast = sim.particles[i_ast]

    # Find the nearest star
    r_min = np.inf
    j_near = None
    for j in star_indices:
        p_star = sim.particles[j]
        dx = p_ast.x - p_star.x
        dy = p_ast.y - p_star.y
        dz = p_ast.z - p_star.z
        r2 = dx * dx + dy * dy + dz * dz
        if r2 < r_min:
            r_min = r2
            j_near = j

    if j_near is None:
        # No stars in the list - no radiation pressure
        return np.zeros(3), 0.0, None

    p_star = sim.particles[j_near]
    r_vec_AU = np.array([p_ast.x - p_star.x, p_ast.y - p_star.y, p_ast.z - p_star.z])
    r_AU = np.linalg.norm(r_vec_AU)

    # Beta for this star and asteroid (computed in SI, distance-independent)
    M_star_Msun = p_star.m
    beta = compute_beta_single_star(M_star_Msun, R_body_m, rho=rho, Q_pr=Q_pr)

    # Grawitacyjne przyspieszenie od tej gwiazdy w jednostkach REBOUND (AU/yr^2)
    # REBOUND uses G=1 in (AU, yr, Msun), so a_grav = -m_star * r_vec / r^3.
    if r_AU == 0.0:
        return np.zeros(3), beta, j_near

    a_grav_vec = -M_star_Msun * r_vec_AU / (r_AU ** 3)

    # Radiation pressure opposes gravity, with magnitude beta * |a_grav|
    a_rad_vec = -beta * a_grav_vec  # znak minus: promieniowanie "odpycha" od gwiazdy

    return a_rad_vec, beta, j_near

## Impact/ejecta generator (currently for Mars)

In [ ]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.constants import M_sun as M_SUN


# ===============================================================
#  Pomocnicze funkcje losowania
# ===============================================================

def _sample_truncated_power_law(x_min, x_max, alpha, n, rng):
    """Draw n samples from a truncated power-law distribution p(x) ~ x^{-alpha}
    over [x_min, x_max] using inverse-CDF sampling."""
    u_rand = rng.uniform(0.0, 1.0, n)
    if abs(alpha - 1.0) < 1e-10:
        return x_min * (x_max / x_min) ** u_rand
    a = 1.0 - alpha
    return (u_rand * (x_max**a - x_min**a) + x_min**a) ** (1.0 / a)


def _random_cone_directions(normal, half_angle_deg, n, rng):
    """Generate n random unit vectors inside a cone with axis `normal`
    and half-angle `half_angle_deg`, uniformly distributed on the spherical cap."""
    half_rad = np.radians(half_angle_deg)
    cos_theta = rng.uniform(np.cos(half_rad), 1.0, n)
    sin_theta = np.sqrt(1.0 - cos_theta**2)
    phi = rng.uniform(0.0, 2.0 * np.pi, n)

    local = np.column_stack([
        sin_theta * np.cos(phi),
        sin_theta * np.sin(phi),
        cos_theta,
    ])

    normal = np.asarray(normal, dtype=float)
    normal /= np.linalg.norm(normal)
    z_axis = np.array([0.0, 0.0, 1.0])

    if np.allclose(normal, z_axis):
        return local
    if np.allclose(normal, -z_axis):
        local[:, 2] *= -1
        return local

    # Macierz obrotu (Rodrigues) z osi z na wektor `normal`
    v = np.cross(z_axis, normal)
    s = np.linalg.norm(v)
    c_val = np.dot(z_axis, normal)
    V = np.array([[0, -v[2], v[1]],
                   [v[2], 0, -v[0]],
                   [-v[1], v[0], 0]])
    R_mat = np.eye(3) + V + V @ V * (1.0 - c_val) / (s**2)
    return (R_mat @ local.T).T


# ===============================================================
#  Default rock variants (ours go here!)
# ===============================================================
DEFAULT_ROCK_VARIANTS = [
    {"name": "basalt",    "density": 3000.0, "albedo": 0.07, "prob": 0.50},
    {"name": "chondrite", "density": 3500.0, "albedo": 0.15, "prob": 0.30},
    {"name": "ice_rich",  "density": 1500.0, "albedo": 0.40, "prob": 0.20},
]


# ===============================================================
#  Main function: create_mars_impact
# ===============================================================

def create_mars_impact(
    sim,
    n_asteroids=100,
    # ---- Geometria uderzenia ----
    impact_normal=None,
    cone_half_angle=60.0,
    # ---- Ejecta velocity ----
    v_min_kms=5.03,
    v_max_kms=20.0,
    alpha_v=2.5,
    # ---- Rozmiar ----
    R_min_m=0.001,
    R_max_m=5.0,
    q_size=2,
    # ---- Density / rock types ----
    rock_variants=None,
    # ---- Rotacja ----
    spin_period_range=(2.0, 20.0),
    obliquity_range=(0.0, 180.0),
    # ---- Size-velocity correlation ----
    size_velocity_corr=True,
    # ---- Star indices (radiation sources) ----
    star_indices=None,
    # ---- Indeks Marsa w symulacji ----
    mars_index=4,
    # ---- Random seed ----
    seed=None,
):
    """
    Tworzy zdarzenie impaktowe na Marsie i dodaje asteroidy (ejecta) do symulacji.

    Parametry
    ---------
    sim               : rebound.Simulation
    n_asteroids       : int — liczba asteroid do wygenerowania
    impact_normal     : (3,) array | None - ejecta cone axis; None = random direction
    cone_half_angle   : float [deg] - ejecta cone half-angle
    v_min_kms         : float [km/s] - minimum velocity (default ~Mars v_esc)
    v_max_kms         : float [km/s] - maximum ejecta velocity
    alpha_v           : float - power-law index of the velocity distribution (>1)
    R_min_m, R_max_m  : float [m] — zakres promieni asteroid
    q_size            : float - power-law index of the size distribution (>1)
    rock_variants     : list[dict] - rock variants {"name","density","albedo","prob"}
    spin_period_range : (min, max) [h] - range of rotation periods
    obliquity_range   : (min, max) [deg] — zakres nachylenia osi obrotu
    size_velocity_corr: bool - anti-correlation between size and velocity
    star_indices      : list[int] | None — indeksy gwiazd w sim.particles
                        (radiation sources, including the Sun). None = [0] (Sun only).
                        To include Gaia stars: [0] + list(range(9, 59))
    mars_index        : int — indeks Marsa w sim.particles
    seed              : int | None — ziarno generatora losowego

    Zwraca
    -------
    asteroid_df : pd.DataFrame - per-asteroid metadata (for ATM and further analysis)
    first_index : int — indeks pierwszej asteroidy w sim.particles
    """
    rng = np.random.default_rng(seed)

    if rock_variants is None:
        rock_variants = DEFAULT_ROCK_VARIANTS
    if star_indices is None:
        star_indices = [0]

    # ── Stan Marsa w chwili wybuchu ──────────────────────────────
    p_mars = sim.particles[mars_index]
    mars_pos = np.array([p_mars.x, p_mars.y, p_mars.z])
    mars_vel = np.array([p_mars.vx, p_mars.vy, p_mars.vz])

    # ── Normalna powierzchni w punkcie uderzenia ────────────────
    if impact_normal is None:
        vec = rng.standard_normal(3)
        impact_normal = vec / np.linalg.norm(vec)
    else:
        impact_normal = np.asarray(impact_normal, dtype=float)
        impact_normal /= np.linalg.norm(impact_normal)

    # -- 1. Sample radii (power law) -----------------------------
    radii_m = _sample_truncated_power_law(R_min_m, R_max_m, q_size, n_asteroids, rng)

    # -- 2. Sample ejecta velocities (power law) -----------------
    velocities_kms = _sample_truncated_power_law(
        v_min_kms, v_max_kms, alpha_v, n_asteroids, rng,
    )

    # ── 3. Antykorelacja: mniejsze → szybsze ───────────────────
    if size_velocity_corr:
        sorted_r = np.sort(radii_m)
        sorted_v = np.sort(velocities_kms)[::-1]
        perm = rng.permutation(n_asteroids)
        radii_m = sorted_r[perm]
        velocities_kms = sorted_v[perm]

    # -- 4. Sample rock type -------------------------------------
    names   = [rv["name"]    for rv in rock_variants]
    dens    = np.array([rv["density"] for rv in rock_variants])
    albedos = np.array([rv["albedo"]  for rv in rock_variants])
    probs   = np.array([rv["prob"]    for rv in rock_variants])
    probs  /= probs.sum()

    idx = rng.choice(len(rock_variants), size=n_asteroids, p=probs)
    rho_arr    = dens[idx]
    albedo_arr = albedos[idx]
    rock_names = [names[i] for i in idx]

    # ── 5. Masa z geometrii: m = (4/3)πR³ρ ─────────────────────
    mass_kg   = (4.0 / 3.0) * np.pi * radii_m**3 * rho_arr
    mass_Msun = mass_kg / M_SUN.value

    # -- 6. Beta - radiation pressure from the nearest star -------
    Q_pr_arr = 1.0 + (2.0 / 3.0) * albedo_arr

    # Find the star from star_indices nearest to the position of Mars
    nearest_star_idx = star_indices[0]
    r2_min = np.inf
    for j in star_indices:
        ps = sim.particles[j]
        dx = mars_pos[0] - ps.x
        dy = mars_pos[1] - ps.y
        dz = mars_pos[2] - ps.z
        r2 = dx*dx + dy*dy + dz*dz
        if r2 < r2_min:
            r2_min = r2
            nearest_star_idx = j

    nearest_star_mass = sim.particles[nearest_star_idx].m
    beta_arr = compute_beta_single_star(
        nearest_star_mass, radii_m, rho=rho_arr, Q_pr=Q_pr_arr,
    )

    # -- 7. Ejecta directions (cone around the normal) -----------
    directions = _random_cone_directions(
        impact_normal, cone_half_angle, n_asteroids, rng,
    )

    # -- 8. Velocities in AU/yr = ejecta + orbital velocity of Mars
    kms_to_au_yr = (1.0 * u.km / u.s).to(u.AU / u.yr).value
    v_au_yr = velocities_kms * kms_to_au_yr

    vx = directions[:, 0] * v_au_yr + mars_vel[0]
    vy = directions[:, 1] * v_au_yr + mars_vel[1]
    vz = directions[:, 2] * v_au_yr + mars_vel[2]

    # ── 9. Parametry rotacji ────────────────────────────────────
    spin_periods_h = rng.uniform(
        spin_period_range[0], spin_period_range[1], n_asteroids,
    )
    obliquities_deg = rng.uniform(
        obliquity_range[0], obliquity_range[1], n_asteroids,
    )
    spin_axes = rng.standard_normal((n_asteroids, 3))
    spin_axes /= np.linalg.norm(spin_axes, axis=1, keepdims=True)

    # -- 10. Add particles to REBOUND ----------------------------
    m_to_au   = (1.0 * u.m).to(u.AU).value
    radii_AU  = radii_m * m_to_au
    first_idx = sim.N

    for i in range(n_asteroids):
        sim.add(
            m=mass_Msun[i],
            r=radii_AU[i],
            x=mars_pos[0], y=mars_pos[1], z=mars_pos[2],
            vx=vx[i], vy=vy[i], vz=vz[i],
        )

    # -- 11. DataFrame with the full metadata --------------------
    asteroid_df = pd.DataFrame({
        "sim_index":     np.arange(first_idx, first_idx + n_asteroids),
        "radius_m":      radii_m,
        "radius_AU":     radii_AU,
        "density_kg_m3": rho_arr,
        "rock_type":     rock_names,
        "albedo":        albedo_arr,
        "mass_kg":       mass_kg,
        "mass_Msun":     mass_Msun,
        "beta":          beta_arr,
        "Q_pr":          Q_pr_arr,
        "nearest_star":  nearest_star_idx,
        "v_ejecta_kms":  velocities_kms,
        "spin_period_h": spin_periods_h,
        "obliquity_deg": obliquities_deg,
        "spin_axis_x":   spin_axes[:, 0],
        "spin_axis_y":   spin_axes[:, 1],
        "spin_axis_z":   spin_axes[:, 2],
    })

    # ── Podsumowanie ────────────────────────────────────────────
    print(f"Dodano {n_asteroids} asteroid (indeksy {first_idx}–{first_idx + n_asteroids - 1})")
    print(f"  Promienie:        {radii_m.min():.4f} – {radii_m.max():.2f} m")
    print(f"  Ejecta velocities: {velocities_kms.min():.2f} - {velocities_kms.max():.2f} km/s")
    print(f"  Beta (gwiazda #{nearest_star_idx}): {beta_arr.min():.2e} – {beta_arr.max():.2e}")
    print(f"  Rock types:        {pd.Series(rock_names).value_counts().to_dict()}")
    print(f"  Spin period:      {spin_periods_h.min():.1f} – {spin_periods_h.max():.1f} h")
    print(f"  Particles total:   {sim.N}")

    return asteroid_df, first_idx

## Test wybuchu

In [ ]:
asteroid_df, first_ast_idx = create_mars_impact(
    sim,
    n_asteroids=200,
    cone_half_angle=55.0,
    v_min_kms=5.03,
    v_max_kms=25.0,
    alpha_v=2.5,
    R_min_m=0.001,
    R_max_m=10.0,
    q_size=1.5,
    spin_period_range=(2.0, 20.0),
    obliquity_range=(0.0, 180.0),
    size_velocity_corr=True,
    star_indices=[0] + list(range(9, 59)),
    seed=42,
)

asteroid_df.head(10)

## Asteroid removal helper

In [ ]:
def remove_all_asteroids(sim, n_permanent=59):
    """Remove every particle except the permanent ones (Sun + planets + Gaia stars).
    
    Default n_permanent=59: 1 Sun + 8 planets + 50 stars.
    """
    n_before = sim.N
    for i in range(sim.N - 1, n_permanent - 1, -1):
        sim.remove(i)
    print(f"Removed {n_before - sim.N} asteroids. {sim.N} particles left.")


# Call:
remove_all_asteroids(sim)

## Radiation pressure (works once asteroids exist)

In [ ]:
import reboundx
import astropy.units as u
from astropy.constants import c as c_light

# 1. Inicjalizacja REBOUNDx
rebx = reboundx.Extras(sim)

# 2. Load the radiation_forces effect
rf = rebx.load_force("radiation_forces")
rebx.add_force(rf)

# 3. Speed of light in simulation units (AU/yr)
rf.params["c"] = c_light.to(u.AU / u.yr).value

# 4. Assign beta to each asteroid
for _, row in asteroid_df.iterrows():
    sim.particles[int(row["sim_index"])].params["beta"] = row["beta"]

print(f"Radiation forces aktywne. c = {rf.params['c']:.2f} AU/yr")
print(f"Beta przypisane {len(asteroid_df)} asteroidom.")

## Cosmic radiation module (based on the contributors' files)

## Co jeszcze:
- choose the integrator and the time step
- the ATM package - e.g. to compute the temperature inside an asteroid (https://github.com/moeyensj/atm)
- pakiet Amuse - "wzbogacenie"
- cosmic radiation, internal radiation, hydrolysis, erosion
- final outcome (capture)
- update the rock parameters in the impact function

## To decide:
- albedo, density and radioactive isotope composition of the rocks